# 2.3 — Méthode par dictionnaire

1. Construire un dictionnaire à partir des mentions annotées de TRAIN.
2. Le chercher dans VALIDATION et régler les choix (casse, filtre, regex).
3. Inspecter 10 sorties, dont des phrases sans entité, puis calculer les scores.

TEST ne sert jamais à enrichir le dictionnaire. Code : `src/dictionnaire.py` (adapté du cours) ; scores : `src/extraction.py`.

In [1]:
import sys, json, random, time
from collections import Counter
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from tals import erreurs
from tals.spans import tokeniser_avec_offsets
from src.cdr import CATEGORIES, charger_jsonl, mentions_reference, decouper_phrases
from src.dictionnaire import DictionnaireCDR, ConfigDictionnaire, REGEX
from src.extraction import tableau_scores, ecrire_predictions

RESULTATS = RACINE / "resultats" / "extraction"
SEED = 42

documents = charger_jsonl(RACINE / "data" / "corpus_extraction.jsonl")  # écrit par 2.2
train = [d for d in documents if d.partition == "train"]
validation = [d for d in documents if d.partition == "validation"]
test = [d for d in documents if d.partition == "test"]
ref_val = mentions_reference(validation)
textes_val = {d.document_id: d.texte for d in validation}
print(len(train), len(validation), len(test), "documents ;", len(ref_val), "mentions de référence en VALIDATION")

500 500 500 documents ; 9591 mentions de référence en VALIDATION


## Construire le dictionnaire à partir de TRAIN

Chaque mention annotée de TRAIN devient une entrée : texte → catégorie. Si un même texte est annoté avec les deux catégories, on garde la plus fréquente.

On calcule aussi le **taux d'annotation** de chaque entrée : quand ce texte apparaît dans TRAIN, à quelle fréquence est-il annoté ? Un taux bas annonce des fausses détections.

In [2]:
base = DictionnaireCDR(ConfigDictionnaire(casse="insensible")).construire(train)
stats = pd.DataFrame([{"forme": " ".join(k), **v} for k, v in base.stats.items()])
print("Entrées :", len(stats))
print(stats["type"].value_counts().to_string())
formes_par_type = {t: {base._cle(m.texte) for d in train for m in d.mentions if m.type == t} for t in CATEGORIES}
ambigues = formes_par_type["Chemical"] & formes_par_type["Disease"]
print("Formes annotées avec les deux types :", len(ambigues), [" ".join(k) for k in sorted(ambigues)][:10])

Entrées : 2390
type
Disease     1384
Chemical    1006
Formes annotées avec les deux types : 1 ['pcp']


In [3]:
# Formes fréquentes dans TRAIN mais rarement annotées : sources probables de faux positifs
stats.query("occurrences >= 5").sort_values("taux").head(15)

,forme,type,annotations,occurrences,taux
1049,mg,Chemical,3,386,0.007772
2133,be,Chemical,7,327,0.021407
564,all,Disease,4,129,0.031008
2151,h,Chemical,3,79,0.037975
471,can,Disease,3,71,0.042254
673,hg,Chemical,1,17,0.058824
1864,acute hepatitis,Disease,1,12,0.083333
221,infarction,Disease,4,48,0.083333
2248,nephritis,Disease,1,11,0.090909
1393,haemorrhage,Disease,1,10,0.100000


## Réglages testés sur VALIDATION

- **Casse** : `exacte` ou `insensible` (« Fentanyl » = « fentanyl »).
- **Filtre** : on retire les entrées dont le taux d'annotation est sous un seuil (0 = pas de filtre).
- **Regex** : terminaisons typiques (Chemical : *-mycin, -cillin, -olol, -pril…* ; Disease : *-itis, -emia, -pathy, -oma…*).
- **Mentions concurrentes** : la plus longue gagne (« renal failure » plutôt que « renal »).

On compare 16 combinaisons.

In [4]:
reglage = []
for casse in ["exacte", "insensible"]:
    for seuil in [0.0, 0.3, 0.5, 0.7]:
        for regex in [False, True]:
            cfg = ConfigDictionnaire(casse, seuil, regex)
            t0 = time.perf_counter()
            dic = DictionnaireCDR(cfg).construire(train)
            pred = dic.extraire_corpus(validation)
            duree = time.perf_counter() - t0
            df = tableau_scores(ref_val, pred, "dictionnaire", "validation", duree).set_index("categorie")
            reglage.append({"casse": casse, "seuil": seuil, "regex": regex, "entrees": len(dic.entrees),
                            "P": df.loc["global (micro)", "precision"], "R": df.loc["global (micro)", "rappel"],
                            "F1": df.loc["global (micro)", "F1"],
                            "F1 Chemical": df.loc["Chemical", "F1"], "F1 Disease": df.loc["Disease", "F1"],
                            "temps_s": round(duree, 1)})
reglage = pd.DataFrame(reglage)
reglage.to_csv(RESULTATS / "dictionnaire_reglage_validation.csv", index=False)
reglage.sort_values("F1", ascending=False)

,casse,seuil,regex,entrees,P,R,F1,F1 Chemical,F1 Disease,temps_s
11,insensible,0.3,True,2348,0.8835,0.6438,0.7449,0.7715,0.7127,2.2
13,insensible,0.5,True,2319,0.8876,0.6340,0.7397,0.7697,0.7029,2.2
1,exacte,0.0,True,2719,0.8675,0.6354,0.7335,0.7554,0.7073,2.7
3,exacte,0.3,True,2683,0.8818,0.6268,0.7328,0.7582,0.7020,2.7
5,exacte,0.5,True,2658,0.8862,0.6180,0.7282,0.7566,0.6934,2.6
10,insensible,0.3,False,2348,0.8903,0.6153,0.7277,0.7557,0.6934,2.2
12,insensible,0.5,False,2319,0.8954,0.6041,0.7215,0.7538,0.6811,2.1
15,insensible,0.7,True,2229,0.8898,0.6055,0.7206,0.7639,0.6663,2.2
0,exacte,0.0,False,2719,0.8690,0.6045,0.7130,0.7352,0.6864,7.6
2,exacte,0.3,False,2683,0.8881,0.5941,0.7119,0.7378,0.6803,2.6


### Les regex sont-elles utiles ?

On compare la meilleure configuration avec et sans regex, et on regarde les mentions ajoutées.

In [6]:
meilleure = reglage.sort_values(["F1", "P"], ascending=False).iloc[0]
cfg_sans = ConfigDictionnaire(meilleure["casse"], float(meilleure["seuil"]), False)
cfg_avec = ConfigDictionnaire(meilleure["casse"], float(meilleure["seuil"]), True)
sans = DictionnaireCDR(cfg_sans).construire(train).extraire_corpus(validation)
avec = DictionnaireCDR(cfg_avec).construire(train).extraire_corpus(validation)
ajout = avec - sans
print(f"Mentions ajoutées par les regex : {len(ajout)}, dont {len(ajout & ref_val)} correctes "
      f"et {len(ajout - ref_val)} fausses")
for type_ in CATEGORIES:
    ok = Counter(textes_val[m.doc][m.debut:m.fin].lower() for m in ajout & ref_val if m.type == type_)
    ko = Counter(textes_val[m.doc][m.debut:m.fin].lower() for m in ajout - ref_val if m.type == type_)
    print(f"\n{type_} — correctes :", ok.most_common(8))
    print(f"{type_} — fausses   :", ko.most_common(8))

Mentions ajoutées par les regex : 361, dont 274 correctes et 87 fausses

Chemical — correctes : [('fluconazole', 10), ('valsartan', 10), ('netilmicin', 9), ('ropivacaine', 8), ('carteolol', 8), ('ketoprofen', 7), ('levobupivacaine', 7), ('piperacillin', 6)]
Chemical — fausses   : [('mitomycin', 8), ('hydroxymetoprolol', 2), ('deoxynojirimycin', 2), ('carteolol', 1), ('cilastatin', 1), ('puromycin', 1), ('fenoprofen', 1), ('piperacillin', 1)]

Disease — correctes : [('hypokalemia', 17), ('uveitis', 11), ('anaemia', 9), ('neuropathy', 8), ('polymyositis', 7), ('arthralgia', 6), ('cytotoxicity', 6), ('neuroblastoma', 5)]
Disease — fausses   : [('neuropathy', 15), ('carcinoma', 9), ('leukemia', 7), ('hyperglycinemia', 4), ('rhinitis', 4), ('lymphoma', 3), ('dystrophy', 3), ('lysozymuria', 3)]


## Configuration retenue et scores sur VALIDATION

On garde la configuration au meilleur F1 (à égalité, la meilleure précision).

In [7]:
CONFIG = ConfigDictionnaire(meilleure["casse"], float(meilleure["seuil"]), bool(meilleure["regex"]))
print("Configuration retenue :", CONFIG.nom())

t0 = time.perf_counter()
dictionnaire = DictionnaireCDR(CONFIG).construire(train)
t_construction = time.perf_counter() - t0
t0 = time.perf_counter()
pred_val = dictionnaire.extraire_corpus(validation)
t_validation = time.perf_counter() - t0
print(f"Construction : {t_construction:.1f} s ; extraction sur VALIDATION : {t_validation:.1f} s")

scores_val = tableau_scores(ref_val, pred_val, "dictionnaire", "validation", t_validation)
scores_val.to_csv(RESULTATS / "scores_dictionnaire_validation.csv", index=False)
ecrire_predictions(pred_val, "dictionnaire", validation, RESULTATS / "predictions_dictionnaire_validation.jsonl")
scores_val

Configuration retenue : casse=insensible, seuil=0.3, regex=oui
Construction : 3.7 s ; extraction sur VALIDATION : 3.4 s


,methode,partition,categorie,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
0,dictionnaire,validation,Chemical,3501,228,1846,0.9389,0.6548,0.7715,False,NaN
1,dictionnaire,validation,Disease,2674,586,1570,0.8202,0.6301,0.7127,False,NaN
2,dictionnaire,validation,global (micro),6175,814,3416,0.8835,0.6438,0.7449,False,3.4


## Inspecter dix sorties

5 phrases avec mention et 5 phrases sans mention, tirées au hasard. Notation : `[mention]<Type>` = attendu, `{mention}<Type>` = prédit, `[{mention}]<Type>` = correct.

In [8]:
def marquer(texte, a, b, ref, pred):
    bornes = []
    for m in ref:
        bornes += [(m.debut, 1, "["), (m.fin, 0, f"]<{m.type[:4]}>")]
    for m in pred:
        bornes += [(m.debut, 2, "{"), (m.fin, -1, f"}}<{m.type[:4]}>")]
    sortie, curseur = [], a
    for pos, _, s in sorted(b_ for b_ in bornes if a <= b_[0] <= b):
        sortie += [texte[curseur:pos], s]
        curseur = pos
    sortie.append(texte[curseur:b])
    return "".join(sortie)

phrases_avec, phrases_sans = [], []
for d in validation:
    for a, b in decouper_phrases(d.texte):
        r = {m for m in ref_val if m.doc == d.document_id and a <= m.debut < b}
        p = {m for m in pred_val if m.doc == d.document_id and a <= m.debut < b}
        (phrases_avec if r else phrases_sans).append((d, a, b, r, p))

rng = random.Random(SEED)
for titre, groupe in [("AVEC mention de référence", phrases_avec), ("SANS mention cible", phrases_sans)]:
    print(f"===== 5 phrases {titre} =====")
    for d, a, b, r, p in rng.sample(groupe, 5):
        print(f"[{d.document_id}] VP={len(r & p)} FP={len(p - r)} FN={len(r - p)}")
        print("  ", marquer(d.texte, a, b, r, p))
    print()

fp_sans = sum(len(p) for *_, r, p in phrases_sans)
print(f"Sur les {len(phrases_sans)} phrases sans mention cible de VALIDATION : {fp_sans} fausses détections")

===== 5 phrases AVEC mention de référence =====
[2528969] VP=1 FP=0 FN=0
   Following a 3-week observation period, volunteers were treated with [{zidovudine}<Chem>]<Chem> 600 mg/day for 18 weeks, 900 mg/day for 9 weeks and 1200 mg/day for 9 weeks, followed by a washout period of 6 weeks after which they were re-started on 1200 mg/day or the highest tolerated dose at 8-hourly intervals.
[9022662] VP=1 FP=0 FN=1
   The children had a medical history significant for [attention deficit disorder]<Dise> previously treated with [{methylphenidate}<Chem>]<Chem> without success.
[20705401] VP=3 FP=1 FN=1
   The symptoms of [{methamphetamine}<Chem>]<Chem> ([{METH}<Chem>]<Chem>)-induced [{psychosis}<Dise>]<Dise> are similar to those of [paranoid type {schizophrenia}<Dise>]<Dise>.
[16132524] VP=0 FP=0 FN=1
   Human malignant [brain tumors]<Dise> have a poor prognosis in spite of surgery and radiation therapy.
[16629641] VP=2 FP=2 FN=2
   [Chronic infection with {hepatitis}<Dise> C virus]<Dise> (HCV

## Types d'erreurs sur VALIDATION

- **frontière** : bonne zone, mauvaises limites ;
- **type** : bonnes limites, mauvaise catégorie ;
- **manquée** : mention non trouvée ;
- **parasite** : détection là où il n'y a pas de mention.

Pour les mentions manquées, on vérifie si elles étaient absentes de TRAIN.

In [9]:
err = erreurs(ref_val, pred_val, textes_val)
print({k: len(v) for k, v in err.items() if k != "surfaces"})

inconnues = [m for m in err["manquee"] if dictionnaire._cle(textes_val[m.doc][m.debut:m.fin]) not in dictionnaire.entrees]
print(f"Mentions manquées dont la forme n'est pas dans le dictionnaire : {len(inconnues)} / {len(err['manquee'])}")

def s(m):
    return textes_val[m.doc][m.debut:m.fin]

print("\nExemples de frontière (référence → prédiction) :")
for att, obt in err["frontiere"][:6]:
    print(f"  {s(att)!r}<{att.type}> → {s(obt)!r}<{obt.type}>")
print("\nExemples de confusion de type :")
for att, obt in err["type"][:6]:
    print(f"  {s(att)!r} : référence {att.type}, prédit {obt.type}")
print("\nFormes manquées les plus fréquentes :", Counter(s(m).lower() for m in err["manquee"]).most_common(10))
print("Parasites les plus fréquents :", Counter(s(m).lower() for m in err["parasite"]).most_common(10))

{'correcte': 6175, 'frontiere': 578, 'type': 13, 'manquee': 2825, 'parasite': 223}
Mentions manquées dont la forme n'est pas dans le dictionnaire : 2808 / 2825

Exemples de frontière (référence → prédiction) :
  'secondary hyperparathyroidism'<Disease> → 'hyperparathyroidism'<Disease>
  'secondary hyperparathyroidism'<Disease> → 'hyperparathyroidism'<Disease>
  'secondary hyperparathyroidism'<Disease> → 'hyperparathyroidism'<Disease>
  'ventricular arrhythmia'<Disease> → 'arrhythmia'<Disease>
  'ventricular arrhythmia'<Disease> → 'arrhythmia'<Disease>
  'renal tubular dysfunction'<Disease> → 'tubular dysfunction'<Disease>

Exemples de confusion de type :
  'CM' : référence Chemical, prédit Disease
  'CM' : référence Chemical, prédit Disease
  'CM' : référence Chemical, prédit Disease
  'CM' : référence Chemical, prédit Disease
  'CM' : référence Chemical, prédit Disease
  'CM' : référence Chemical, prédit Disease

Formes manquées les plus fréquentes : [('dexmedetomidine', 20), ('no', 1

## Gel et prédictions sur TEST

La configuration est figée. Le dictionnaire, construit sur TRAIN, est appliqué aux textes de TEST ; le score TEST est calculé dans `5_evaluation.ipynb`. On sauvegarde le dictionnaire, la configuration, les prédictions et les temps.

In [10]:
t0 = time.perf_counter()
pred_test = dictionnaire.extraire_corpus(test)
t_test = time.perf_counter() - t0
ecrire_predictions(pred_test, "dictionnaire", test, RESULTATS / "predictions_dictionnaire_test.jsonl")

pd.DataFrame([{"forme": " ".join(k), "categorie": t, **{c: v for c, v in dictionnaire.stats[k].items() if c != "type"}}
              for k, t in sorted(dictionnaire.entrees.items())]
             ).to_csv(RESULTATS / "dictionnaire.tsv", sep="\t", index=False)

gel = {"config": vars(CONFIG), "entrees": len(dictionnaire.entrees),
       "regex": {k: v.pattern for k, v in REGEX.items()} if CONFIG.regex else None,
       "temps_s": {"construction_train": round(t_construction, 2), "extraction_validation": round(t_validation, 2),
                   "extraction_test": round(t_test, 2)},
       "predictions_test": len(pred_test)}
(RESULTATS / "dictionnaire_config_gelee.json").write_text(json.dumps(gel, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(gel, ensure_ascii=False, indent=2))

{
  "config": {
    "casse": "insensible",
    "seuil_annotation": 0.3,
    "regex": true
  },
  "entrees": 2348,
  "regex": {
    "Chemical": "\\b[A-Za-z][a-z]{2,}(?:mycin|micin|cillin|cycline|azole|olol|pril|sartan|platin|statin|tidine|azepam|azolam|caine|profen|oxacin|vudine|navir|parin|semide|thiazide|triptyline|pramine|peridol|zodone|xetine|barbital)\\b",
    "Disease": "\\b[A-Za-z][a-z]{2,}(?:itis|emia|aemia|pathy|toxicity|algia|plegia|penia|cytosis|trophy|oma|uria)\\b"
  },
  "temps_s": {
    "construction_train": 1.4,
    "extraction_validation": 1.39,
    "extraction_test": 1.52
  },
  "predictions_test": 7019
}
